# Synthetic Cohort Generator — Demo

Generates synthetic patient cohorts from the three example disease configs in `../examples/` and visualizes:

1. The marginal distribution of each baseline feature.
2. The empirical correlation matrix.
3. The subtype proportions.

Run all cells top-to-bottom.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from cohort_generator import generate, load_config

EXAMPLES = Path('..') / 'examples'
CONFIGS = {
    'metabolic_t2d_like': EXAMPLES / 'metabolic_t2d_like.json',
    'neurodegenerative_ad_like': EXAMPLES / 'neurodegenerative_ad_like.json',
    'cardiovascular_chf_like': EXAMPLES / 'cardiovascular_chf_like.json',
}

In [ ]:
def show_marginals(df, config, n_cols=4):
    features = config.baseline_features
    n_rows = int(np.ceil(len(features) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3, n_rows * 2.5))
    axes = np.atleast_2d(axes).ravel()
    for ax, feat in zip(axes, features):
        col = df[feat]
        if col.dtype == 'O':
            col.value_counts().plot(kind='bar', ax=ax, color='steelblue')
        else:
            ax.hist(col, bins=30, color='steelblue', edgecolor='white')
        ax.set_title(feat, fontsize=9)
    for ax in axes[len(features):]:
        ax.set_visible(False)
    fig.suptitle(f'Marginals — {config.disease_name}', y=1.02)
    fig.tight_layout()
    plt.show()


def show_correlation(df, config):
    cont_features = [
        f for f in config.baseline_features
        if config.baseline_distributions[f].type in ('normal', 'lognormal')
    ]
    corr = df[cont_features].corr()
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(corr, vmin=-1, vmax=1, cmap='RdBu_r')
    ax.set_xticks(range(len(cont_features)))
    ax.set_yticks(range(len(cont_features)))
    ax.set_xticklabels(cont_features, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(cont_features, fontsize=8)
    fig.colorbar(im, ax=ax, shrink=0.7)
    ax.set_title(f'Pearson correlation — {config.disease_name}')
    fig.tight_layout()
    plt.show()


def show_subtypes(df, config):
    counts = df['subtype'].value_counts(normalize=True)
    fig, ax = plt.subplots(figsize=(6, 3))
    x = np.arange(len(config.subtypes))
    target = [s.prevalence for s in config.subtypes]
    names = [s.name for s in config.subtypes]
    empirical = [counts.get(n, 0.0) for n in names]
    width = 0.4
    ax.bar(x - width/2, target, width, label='target', color='lightgray')
    ax.bar(x + width/2, empirical, width, label='empirical', color='steelblue')
    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=30, ha='right', fontsize=8)
    ax.set_ylabel('proportion')
    ax.set_title(f'Subtype proportions — {config.disease_name}')
    ax.legend()
    fig.tight_layout()
    plt.show()

In [ ]:
for name, path in CONFIGS.items():
    print(f'\n=== {name} ===')
    config = load_config(path)
    df = generate(config, n=1000, seed=42)
    print(df.head())
    show_marginals(df, config)
    show_correlation(df, config)
    show_subtypes(df, config)